<a href="https://colab.research.google.com/github/visionbyangelic/Neuro-Data/blob/main/Multimodal-Brain-Encoding-FriendsTvShow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Part 1 — What this data actually *is*

## The scanner

A person lies in an fMRI scanner and watches *Friends*. Every **1.49 seconds**, the scanner takes one snapshot of their whole brain.

That snapshot is called a **TR**. It's just a moment in time.

The brain gets carved into **1000 little regions**, called **parcels** (using something called the Schaefer atlas — think of it as a postcode system for the brain). So each snapshot gives us **1000 numbers**: how active each region was at that moment.

A 14-minute episode = ~590 snapshots. So the brain data for one episode is a table:

```
          parcel_0  parcel_1  parcel_2  ...  parcel_999
TR 0        0.31      -0.22      1.04    ...    0.09
TR 1        0.28      -0.19      0.97    ...    0.11
TR 2       -0.05       0.41      0.88    ...   -0.30
...
TR 590      ...
```

**590 rows × 1000 columns.** That's it. That's the brain.

## The movie

Now — you can't feed an `.mkv` video into a regression. Maths needs numbers.

So somebody ran the episode through three AI models ahead of time:

| What went in | What came out |
|---|---|
| the **video frames** | **250 numbers** per moment (describing what's on screen) |
| the **soundtrack** | **20 numbers** per moment (describing what it sounds like) |
| the **transcript** | **250 numbers** per moment (describing what was said) |

This is called **feature extraction**, and it's already done for us.

Crucially, these come out **at the same rate as the brain scans** — one row per TR. So the movie is *also* a table with ~590 rows.

## X and Y

Two tables. Same number of rows. Row 5 of one and row 5 of the other are **the same moment in time.**

- **X = the movie** (what we feed in). 250 + 20 + 250 = **520 columns**
- **Y = the brain** (what we're predicting). **1000 columns**

## The encoding model

We build a **guesser**. It reads X and tries to output Y.

1. **Train:** show it thousands of moments — *"the movie looked like this, and the brain did that."* It learns the mapping.
2. **Test:** show it moments it has **never seen**. Give it only the movie. Ask it to guess the brain.
3. **Score:** how close was the guess? For each of the 1000 parcels, we get a number **r** (a correlation). r = 0 means useless. r = 0.5 means quite good.

## Our experiment, in one sentence

Build the guesser **twice**:

- **Baseline:** knows only picture + sound
- **Full:** knows picture + sound + **the words that were spoken**

Then ask: **does the Full guesser get better *specifically* in the language areas** — and not everywhere?

That word *specifically* is the entire experiment. If the words help everywhere equally, we've learned nothing about language; we've just built a slightly better guesser.

---

# Part 2 — What we thought vs. what we've got

Your team's founding belief:

> *"We got season 1 episode 1 only — we won't have enough data."*

**It was wrong.** Here's why.

Inside `stimulus_features/` there are **two** folders:

```
stimulus_features/
├── raw/    ← ONE episode.  s01e01a only.        ⚠️ THE DEMO
└── pca/    ← 292 episodes. Six whole seasons.   ⭐ THE REAL DATASET
```

**We kept opening `raw/`.** It's the tutorial's worked example — *"here's what feature extraction produces, using one episode."* It was never the dataset.

`pca/friends/` was one folder over. The whole time.

| | We thought | Reality |
|---|---|---|
| Episodes | 1 | **282 clean** |
| Timepoints to train on | 591 | **~133,000** |
| Subjects | 4 | 4 ✅ |
| Features per modality | unknown, feared unequal | visual **250**, language **250**, audio **20** |
| Can we answer the language question? | *"probably too noisy"* | **Yes, comfortably** |

**The tell was the file sizes.** `raw/visual` is 19 MB. `pca/visual` is **137 MB**. But PCA makes data *smaller* — that's its whole job. A "compressed" file that's 7× bigger can only mean one thing: **far more timepoints inside.**

---

# Part 3 — The code, chunk by chunk

### Chunk 1 — Find where the data lives

```python
import glob
hits = glob.glob('/content/drive/MyDrive/**/fmri', recursive=True)
print(hits)
```

**What it does:** searches your whole Drive for any folder called `fmri`. The `**` means "any number of folders deep."

**Why:** instead of typing a path and hoping, let Python find it.

---

### Chunk 2 — Set paths, and **assert** them

```python
import os
BASE = '/content/drive/MyDrive/Neuromatch/algonauts_2025_challenge_tutorial_data'
ROOT = f'{BASE}/algonauts_2025.competitors'   # the BRAIN
SF   = f'{BASE}/stimulus_features'            # the MOVIE

assert os.path.exists(ROOT), ROOT
assert os.path.exists(SF),   SF
```

**What `assert` does:** if the path doesn't exist, it **crashes loudly**.

**Why this matters more than it looks:** if you hand `os.walk()` a path that doesn't exist, it **doesn't error**. It silently returns nothing. Your loop prints nothing. You conclude the folder is empty.

*That is literally how we convinced ourselves the feature files didn't exist.* They did.

> 🔑 **Loud bugs are the good bugs.** `FileNotFoundError` stops you. Silent bugs hand you numbers, and the numbers look fine.

---

### Chunk 3 — Map the folder

```python
for dirpath, dirnames, filenames in os.walk(BASE):
    depth = dirpath.replace(BASE, '').count('/')
    print('   ' * depth, os.path.basename(dirpath) + '/')
    for f in sorted(filenames):
        mb = os.path.getsize(os.path.join(dirpath, f)) / 1e6
        print('   ' * (depth+1), f, f'({mb:.1f} MB)')
```

**What it does:** crawls every folder, prints the tree with file sizes.

**Why:** this one cell is what exposed the mistake. It shows `raw/` **and** `pca/` side by side, with their sizes.

---

### Chunk 4 — Load X (the movie)

```python
import numpy as np
P = f'{SF}/pca/friends'
mods = {m: np.load(f'{P}/{m}/features_train.npy', allow_pickle=True).item()
        for m in ['visual', 'audio', 'language']}
```

**What's in the file:** not a plain table — a **dictionary**:

```python
{'s01e01a': array(591 × 250),
 's01e01b': array(590 × 250),
 's01e02a': array(...),
 ... 292 episodes ...}
```

**`allow_pickle=True`** — needed to unpack a dictionary from a `.npy` file.
**`.item()`** — unwraps the weird 0-dimensional box numpy puts it in.

**Why `mmap_mode` failed earlier:** memory-mapping only works on plain number-grids. This is a dict of objects. That `ValueError` was actually our **first clue** — a dict means *many episodes keyed by name*.

---

### Chunk 5 — Check the shapes

```python
for m in ['visual', 'audio', 'language']:
    dm    = mods[m]
    total = sum(np.asarray(v).shape[0] for v in dm.values())
    dim   = np.asarray(dm['s01e01a']).shape[-1]
    print(f'{m:9s} {len(dm):4d} episodes  {total:7d} TRs  {dim:4d} dims')
```

Output:
```
visual     292 episodes   137687 TRs   250 dims
audio      292 episodes   137687 TRs    20 dims
language   292 episodes   137681 TRs   250 dims
```

**Three things to read:**
- **292 episodes.** Not 1. 🎉
- **Visual and language are both 250.** We'd worried a huge visual block would drown out a small language block. **That worry is dead** — someone equalised them for us.
- **Language is 6 TRs short.** `137687 − 137681 = 6`. We chase that down below.

---

### Chunk 6 — Load Y (the brain)

```python
import h5py
func_path = glob.glob(f'{ROOT}/fmri/sub-03/func/*task-friends*.h5')[0]

with h5py.File(func_path, 'r') as f:
    ykeys = list(f.keys())
    print(len(ykeys), 'episodes')
    print(f[ykeys[0]].shape)     # (TRs, 1000)
```

**An `.h5` is a folder inside a file** — named keys, each holding a table.

**h5py is lazy.** It reads *only what you ask for*. `f['s01e01a'][:]` pulls ~2 MB. The `[:]` is what actually loads it.

🚫 **Never load the whole file.** It's 488 MB and it will crash Colab.

**The filename is a message:** `desc-s123456` = seasons **1,2,3,4,5,6**. Not a serial number.

---

### Chunk 7 — Match the keys

```python
import re
def ep(k):
    m = re.search(r'task-(s\d{2}e\d{2}[a-z])$', k)
    return m.group(1) if m else None

y2x = {k: ep(k) for k in ykeys}
assert all(y2x.values())
x2y = {v: k for k, v in y2x.items()}
shared = sorted(set(y2x.values()) & set(mods['visual']))
```

**The problem:** the two datasets use different names for the same episode.

- X says → `s01e02a`
- Y says → `ses-001_task-s01e02a`

**What the regex does:** cuts the episode ID out of the long name. Read `task-(s\d{2}e\d{2}[a-z])$` as: *"find `task-`, then grab `s` + 2 digits + `e` + 2 digits + a letter, at the end."*

**The `assert`:** if the naming ever changes, we hear about it **immediately** instead of silently losing episodes.

---

### Chunk 8 — Find the mismatches

```python
from collections import Counter
with h5py.File(func_path, 'r') as f:
    diffs = {}
    for e in shared:
        nx = np.asarray(mods['visual'][e]).shape[0]   # movie rows
        ny = f[x2y[e]].shape[0]                       # brain rows
        if nx != ny:
            diffs[e] = (nx, ny)

print(Counter(ny - nx for nx, ny in diffs.values()))
# → Counter({1: 220, 2: 2})
```

**What it says:** in 220 episodes the **brain has 1 more row than the movie**. In 2 episodes, 2 more. 68 match exactly.

**What that means:** this is a **rounding rule**, not corruption. Almost certainly, the feature extractor drops an incomplete final chunk while the scanner keeps its last scan. The episodes that match are the ones where the division came out even.

---

### Chunk 9 — Drop the weird ones

```python
BAD_TR   = {'s01e01b', 's04e23a'}                                          # brain 2 TRs longer
BAD_LANG = {'s01e03b','s02e20a','s03e04b','s03e17b','s05e23d','s06e23a'}   # language 1 TR shorter
episodes = sorted(set(shared) - BAD_TR - BAD_LANG)
print(len(episodes))   # 282
```

We throw away 8 odd episodes. We lose **3%** of the data and buy **certainty**. Take that trade every time.

---

# Part 4 — The one thing that can still ruin everything

The brain table has **one more row** than the movie table. To stack them, we have to cut one row off. **But which end?**Our code currently assumes **trim from the end**:

```python
n = min(len(Xv), len(Xa), len(Xl), len(Y))
return Xv[:n], Xa[:n], Xl[:n], Y[:n]     # ← cuts the LAST row
```

**That is an assumption, not a fact.** And if it's wrong, ~1.5 seconds of misalignment gets smeared across all 133,000 timepoints — invisibly, in exactly the fine-grained language signal we're hunting for.

## How to settle it — let visual cortex tell you

Here's the trick. **Visual cortex responds to what's on screen.** That is a *fact*, not a hypothesis. So you can use it as a **ruler to check your own pipeline**.

Fit `visual features → visual parcels`, and try **both** alignments:

| trim | delay | r in visual cortex |
|---|---|---|
| end | 0, 1, 2, 3, 4, 5, 6 | ? |
| start | 0, 1, 2, 3, 4, 5, 6 | ? |

**Whichever gives the higher r is the correct one.** Done.

And you get a bonus. Brain activity is *slow* — blood flow peaks about 5 seconds after something happens on screen. That's the **HRF delay**. So the `r` should peak at a delay of ~3–4 TRs. If it does, your pipeline is **provably correct** and you've *measured* the lag instead of guessing it. If the curve is flat, something upstream is broken — and you found it today, not in Week 3.

This is the **smoke test**. It's not a formality. It's an instrument.

---

# Part 5 — What we can now ask

Here's the payoff. With 591 timepoints, most of these were impossible. With 133,000, they're all on the table.

## 🎯 The main question — now actually affordable

> **How well do visual and auditory features alone predict the language network? And when we add the words, does prediction improve *more* in language parcels than in auditory or visual parcels?**

**Unchanged.** It was always the right question. It just stopped being too noisy to answer.

**Auditory cortex is your adversary, not your fallback.** Visual cortex is a gimme — of course the transcript won't predict V1. The comparison that can genuinely **falsify** you is auditory cortex, **because speech *is* sound**. If the words help there too, your "language" features are functionally a speech-detector. That's a real finding, and a deflating one.

## 🆕 Now possible #1 — hold out whole episodes

Train on **seasons 1–5**. Test on **season 6**.

Different scenes, different dialogue, different guest actors. A model that generalises *there* has learned something about how brains process speech — not about the acoustics of one coffee-shop set. Far stronger than any within-episode split.

## 🆕 Now possible #2 — the learning curve *(my favourite, and it's your story)*

Fit the model on **1 episode**, then 5, then 20, then 100, then 282. Plot **r vs. how much data you trained on**, separately for each region.

Why this is beautiful: **it dramatises exactly what happened to you.** You'd be able to stand up in Week 3 and say:

> *"With one episode, the language effect is invisible. Here's the plot. We nearly concluded it wasn't there. It takes ~50 episodes before it emerges."*

That's an honest, simple, interpretable analysis — and it's a genuinely good scientific point about statistical power. It's also a *story*, which is what the final presentation is actually graded on.

## 🆕 Now possible #3 — does it transfer across people?

You have 4 subjects, all in the same 1000-parcel space. Parcel #412 is parcel #412 in every brain.

So: **fit on subjects 1, 2, 3 → predict subject 5.** If the language effect survives, you've found something that isn't idiosyncratic to one person — a shared code.

*(Gemini told you this was "massively complex." With parcellated data it is not. It's one extra loop.)*

This connects directly to one of NMA's **big ideas**: *generalization across individuals*. Good hook for the abstract.

## 🆕 Now possible #4 — does it transfer to a different movie?

You also have **`movie10`** — a completely different set of films, with both the features and the brain data.

**Train on Friends. Test on movie10.** Does the language effect hold on a stimulus the model has never seen the genre of? That's a serious test of generality, and almost no NMA group will do it.

## ⚠️ Still required, regardless — the noise ceiling

Your hypothesis compares *regions*. But regions differ in how noisy they are.

Say you get **language Δr = 0.02, auditory Δr = 0.06.** Two possible worlds:

- **World A:** language parcels are reliable; the words genuinely don't help. **Hypothesis wrong.**
- **World B:** language parcels are just noisier. Their *maximum possible* score is 0.05 — so 0.02 is **40% of everything available**. Auditory's max is 0.60, so 0.06 is **10%**. The words helped **far more** in language, proportionally. **Hypothesis right.**

**These two worlds produce identical plots.** Without a ceiling, you'd pick the wrong one.

**The fix costs nothing.** Four people watched the same episodes. So for each parcel, correlate its timecourse **across subjects** (inter-subject correlation, ISC):

- **High ISC** → all four brains did the same thing at the same moment. That parcel reliably tracks the movie. **Predictable in principle.**
- **Near-zero ISC** → mostly noise. **Nothing could predict it.** Not you, not the Algonauts winners, not a perfect model.

Then your result becomes *"language parcels reached 40% of their ceiling; auditory reached 10% of theirs"* — **interpretable**, instead of two bare numbers nobody can read.

ISC needs **no movie features at all**. Just brain data and a shared stimulus. You have six seasons of both.

---

## What to do next, in order

1. **Someone reads the tutorial's alignment code** and can explain the trim + HRF delay **out loud.** Not "the tutorial handles it." *Explain it.*
2. **Run the smoke test.** Visual → visual cortex, sweeping trim × delay. One figure gives you the alignment *and* the HRF lag.
3. **Write the three parcel lists into the doc** — language / auditory / visual — **before** looking at a single result. Otherwise you're picking the regions *because* the words helped there, then "discovering" that the words helped there. That's circular, and it produces a "result" even from pure noise.
4. **Then, and only then**, fit the three models.

In [15]:
import glob
hits = glob.glob('/content/drive/MyDrive/**/fmri', recursive=True)
print(hits)

['/content/drive/MyDrive/Neuromatch/algonauts_2025_challenge_tutorial_data/algonauts_2025.competitors/fmri']


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import os

roots = ['/content/drive/MyDrive/Neuromatch/algonauts_2025_challenge_tutorial_data/algonauts_2025.competitors',
         '/content/drive/MyDrive/Algonauts/algonauts_2025.competitors',
         '/content/drive/MyDrive/neuromatch/algonauts_2025_challenge_tutorial_data/algonauts_2025.competitors'] # Added a common alternative path

for R in roots:
    print('=' * 70)
    print(R)
    n_files, total = 0, 0
    try:
        # os.walk will yield nothing if R doesn't exist, which is fine for n_files and total
        for dirpath, _, filenames in os.walk(R):
            for f in filenames:
                n_files += 1
                total += os.path.getsize(os.path.join(dirpath, f))
        print(f'{n_files} files, {total/1e9:.2f} GB')
        print('top level:', sorted(os.listdir(R)))
    except FileNotFoundError:
        print(f"Warning: Path not found for {R}. Skipping this root.")
    except Exception as e: # Catch other potential errors
        print(f"An unexpected error occurred for path {R}: {e}. Skipping this root.")

/content/drive/MyDrive/Neuromatch/algonauts_2025_challenge_tutorial_data/algonauts_2025.competitors
22 files, 3.00 GB
top level: ['fmri', 'stimuli']
/content/drive/MyDrive/Algonauts/algonauts_2025.competitors
0 files, 0.00 GB
/content/drive/MyDrive/neuromatch/algonauts_2025_challenge_tutorial_data/algonauts_2025.competitors
0 files, 0.00 GB


In [18]:
ROOT = roots[0]          # or [1], per the above
assert os.path.exists(ROOT)

for dirpath, dirnames, filenames in os.walk(ROOT):
    depth = dirpath.replace(ROOT, '').count('/')
    print('  ' * depth, os.path.basename(dirpath) + '/')
    for f in sorted(filenames):
        size = os.path.getsize(os.path.join(dirpath, f)) / 1e6
        print('  ' * (depth + 1), f, f'({size:.1f} MB)')

 algonauts_2025.competitors/
   fmri/
     sub-05/
       func/
         sub-05_task-friends_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-s123456_bold.h5 (509.8 MB)
         sub-05_task-movie10_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_bold.h5 (92.2 MB)
       atlas/
         sub-05_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-dseg_parcellation.nii.gz (0.1 MB)
       target_sample_number/
         sub-05_friends-s7_fmri_samples.npy (0.0 MB)
         sub-05_ood_fmri_samples.npy (0.0 MB)
     sub-02/
       func/
         sub-02_task-friends_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-s123456_bold.h5 (514.9 MB)
         sub-02_task-movie10_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_bold.h5 (92.3 MB)
       atlas/
         sub-02_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-dseg_parcellation.nii.gz (0.1 MB)
       target_sample_number/
         sub-02_friends-s7_

In [19]:
import os
SF = '/content/drive/MyDrive/Neuromatch/algonauts_2025_challenge_tutorial_data/stimulus_features'
assert os.path.exists(SF), SF

for dirpath, dirnames, filenames in os.walk(SF):
    print(dirpath.replace(SF, ''), '→', len(filenames), 'files')
    for f in sorted(filenames)[:5]:
        print('   ', f, f'({os.path.getsize(os.path.join(dirpath,f))/1e6:.1f} MB)')

 → 0 files
/raw → 0 files
/raw/visual → 1 files
    friends_s01e01a_features_visual.h5 (19.4 MB)
/raw/audio → 1 files
    friends_s01e01a_features_audio.h5 (0.0 MB)
/raw/language → 1 files
    friends_s01e01a_features_language.h5 (20.0 MB)
/pca → 0 files
/pca/friends_movie10 → 0 files
/pca/friends_movie10/visual → 2 files
    features_test.npy (23.5 MB)
    features_train.npy (155.6 MB)
/pca/friends_movie10/language → 2 files
    features_test.npy (23.5 MB)
    features_train.npy (155.6 MB)
/pca/friends_movie10/audio → 2 files
    features_test.npy (1.9 MB)
    features_train.npy (12.5 MB)
/pca/ood → 0 files
/pca/ood/visual → 1 files
    features_ood.npy (4.9 MB)
/pca/ood/language → 1 files
    features_ood.npy (4.0 MB)
/pca/ood/audio → 1 files
    features_ood.npy (0.4 MB)
/pca/friends → 0 files
/pca/friends/audio → 1 files
    features_train.npy (11.0 MB)
/pca/friends/language → 1 files
    features_train.npy (137.7 MB)
/pca/friends/visual → 1 files
    features_train.npy (137.7 MB)


In [20]:
import numpy as np

P = f'{SF}/pca/friends'          # SF is already defined from the walk

for mod in ['visual', 'audio', 'language']:
    # Load the .npy file, allowing pickle to handle Python objects (like dictionaries)
    # Then, call .item() to extract the dictionary from the 0-dimensional NumPy array wrapper.
    loaded_data = np.load(f'{P}/{mod}/features_train.npy', allow_pickle=True).item()

    print(f'{mod}:')
    print(f'  Type of loaded object: {type(loaded_data)}')
    print(f'  Number of episodes: {len(loaded_data)}')
    if len(loaded_data) > 0:
        first_episode_key = list(loaded_data.keys())[0]
        first_episode_array = loaded_data[first_episode_key]
        print(f'  Shape of first episode ({first_episode_key}): {np.asarray(first_episode_array).shape}')
        print(f'  Dtype of first episode data: {np.asarray(first_episode_array).dtype}')
    print('-' * 20)

visual:
  Type of loaded object: <class 'dict'>
  Number of episodes: 292
  Shape of first episode (s01e01a): (591, 250)
  Dtype of first episode data: float32
--------------------
audio:
  Type of loaded object: <class 'dict'>
  Number of episodes: 292
  Shape of first episode (s01e01a): (591, 20)
  Dtype of first episode data: float32
--------------------
language:
  Type of loaded object: <class 'dict'>
  Number of episodes: 292
  Shape of first episode (s01e01a): (591, 250)
  Dtype of first episode data: float32
--------------------


In [21]:
import numpy as np

P = f'{SF}/pca/friends'

X = np.load(f'{P}/visual/features_train.npy', allow_pickle=True)
print(type(X), X.shape, X.dtype)

d = X.item() if X.shape == () else X    # 0-d object array → unwrap to the dict
print(type(d))
print('n keys:', len(d))
print('first keys:', list(d)[:8])

k = list(d)[0]
print('one episode:', k, np.asarray(d[k]).shape)

<class 'numpy.ndarray'> () object
<class 'dict'>
n keys: 292
first keys: ['s01e01a', 's01e01b', 's01e02a', 's01e02b', 's01e03a', 's01e03b', 's01e04a', 's01e04b']
one episode: s01e01a (591, 250)


In [22]:
for mod in ['visual', 'audio', 'language']:
    d = np.load(f'{P}/{mod}/features_train.npy', allow_pickle=True).item()
    total = sum(np.asarray(v).shape[0] for v in d.values())
    dim   = np.asarray(d[list(d)[0]]).shape[-1]
    print(f'{mod:9s} {len(d):4d} episodes   {total:7d} TRs   {dim:4d} dims')

visual     292 episodes    137687 TRs    250 dims
audio      292 episodes    137687 TRs     20 dims
language   292 episodes    137681 TRs    250 dims


In [23]:
FMRI = f'{ROOT}/fmri'   # ROOT = .../algonauts_2025.competitors
import glob
func_path = glob.glob(f'{FMRI}/sub-03/func/*task-friends*.h5')[0]
print(func_path)

/content/drive/MyDrive/Neuromatch/algonauts_2025_challenge_tutorial_data/algonauts_2025.competitors/fmri/sub-03/func/sub-03_task-friends_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-s123456_bold.h5


In [24]:
import h5py, numpy as np
with h5py.File(func_path, 'r') as f:
    ykeys = set(f.keys())
    print('func episodes:', len(ykeys))
    common = sorted(set(d) & ykeys)
    print('shared:', len(common))
    print('X-only:', sorted(set(d) - ykeys)[:5])
    print('Y-only:', sorted(ykeys - set(d))[:5])
    for k in common[:5]:
        print(k, 'X:', np.asarray(d[k]).shape, ' Y:', f[k].shape)

func episodes: 290
shared: 0
X-only: ['s01e01a', 's01e01b', 's01e02a', 's01e02b', 's01e03a']
Y-only: ['ses-001_task-s01e02a', 'ses-001_task-s01e02b', 'ses-001_task-s01e03a', 'ses-001_task-s01e03b', 'ses-002_task-s01e04a']


In [25]:
mods = {}
for mod in ['visual', 'audio', 'language']:
    mods[mod] = np.load(f'{P}/{mod}/features_train.npy', allow_pickle=True).item()

for k in mods['visual']:
    nv = np.asarray(mods['visual'][k]).shape[0]
    nl = np.asarray(mods['language'][k]).shape[0]
    if nv != nl:
        print(k, 'visual:', nv, 'language:', nl, 'diff:', nv - nl)

s01e03b visual: 472 language: 471 diff: 1
s02e20a visual: 449 language: 448 diff: 1
s03e04b visual: 473 language: 472 diff: 1
s03e17b visual: 453 language: 452 diff: 1
s05e23d visual: 488 language: 487 diff: 1
s06e23a visual: 486 language: 485 diff: 1


In [26]:
BAD = {'s01e03b','s02e20a','s03e04b','s03e17b','s05e23d','s06e23a'}
episodes = [k for k in mods['visual'] if k not in BAD]

In [27]:
import re

with h5py.File(func_path, 'r') as f:
    ykeys = list(f.keys())

# 'ses-001_task-s01e02a'  ->  's01e02a'
def ep(k):
    m = re.search(r'task-(s\d{2}e\d{2}[a-z])$', k)
    return m.group(1) if m else None

y2x = {k: ep(k) for k in ykeys}
assert all(y2x.values()), [k for k, v in y2x.items() if v is None]   # loud if the regex misses

xkeys = set(mods['visual'])
shared = sorted(v for v in y2x.values() if v in xkeys)
print('shared:', len(shared))
print('in Y but not X:', sorted(set(y2x.values()) - xkeys))
print('in X but not Y:', sorted(xkeys - set(y2x.values())))

shared: 290
in Y but not X: []
in X but not Y: ['s05e20a', 's06e03a']


In [28]:
x2y = {v: k for k, v in y2x.items()}
with h5py.File(func_path, 'r') as f:
    diffs = {}
    for e in shared:
        nx = np.asarray(mods['visual'][e]).shape[0]
        ny = f[x2y[e]].shape[0]
        if nx != ny:
            diffs[e] = (nx, ny)
    print('episodes where X != Y:', len(diffs))
    print(list(diffs.items())[:10])

episodes where X != Y: 222
[('s01e01a', (591, 592)), ('s01e01b', (590, 592)), ('s01e04a', (502, 503)), ('s01e04b', (502, 503)), ('s01e07a', (492, 493)), ('s01e07b', (492, 493)), ('s01e08a', (475, 476)), ('s01e08b', (475, 476)), ('s01e09a', (467, 468)), ('s01e09b', (467, 468))]


In [29]:
from collections import Counter
print(Counter(ny - nx for nx, ny in diffs.values()))

Counter({1: 220, 2: 2})


In [30]:
def get_xy(ep, f, delay=0):
    Xv = np.asarray(mods['visual'][ep])
    Xa = np.asarray(mods['audio'][ep])
    Xl = np.asarray(mods['language'][ep])
    Y  = f[x2y[ep]][:]
    n = min(len(Xv), len(Xa), len(Xl), len(Y))
    return Xv[:n], Xa[:n], Xl[:n], Y[:n]      # ← truncating from the END. An assumption. Verify it.

In [31]:
print([(e, nx, ny) for e, (nx, ny) in diffs.items() if ny - nx == 2])

[('s01e01b', 590, 592), ('s04e23a', 503, 505)]


In [32]:
BAD_TR   = {'s01e01b', 's04e23a'}                                    # off-by-2
BAD_LANG = {'s01e03b','s02e20a','s03e04b','s03e17b','s05e23d','s06e23a'}  # language short by 1
episodes = sorted(set(shared) - BAD_TR - BAD_LANG)
print(len(episodes), 'clean episodes')                                # ~282

282 clean episodes


# loading atlas

In [33]:
!pip install nilearn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [36]:
import glob
glob.glob('/content/drive/MyDrive/**/*.nii.gz', recursive=True)

['/content/drive/MyDrive/Neuromatch/algonauts_2025_challenge_tutorial_data/algonauts_2025.competitors/fmri/sub-05/atlas/sub-05_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-dseg_parcellation.nii.gz',
 '/content/drive/MyDrive/Neuromatch/algonauts_2025_challenge_tutorial_data/algonauts_2025.competitors/fmri/sub-02/atlas/sub-02_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-dseg_parcellation.nii.gz',
 '/content/drive/MyDrive/Neuromatch/algonauts_2025_challenge_tutorial_data/algonauts_2025.competitors/fmri/sub-01/atlas/sub-01_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-dseg_parcellation.nii.gz',
 '/content/drive/MyDrive/Neuromatch/algonauts_2025_challenge_tutorial_data/algonauts_2025.competitors/fmri/sub-03/atlas/sub-03_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-dseg_parcellation.nii.gz']

In [37]:
from nilearn import plotting, image

atlas_path = '/content/drive/MyDrive/Neuromatch/Algonauts/algonauts_2025.competitors/fmri/sub-01/atlas/sub-01_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-dseg_parcellation.nii.gz'

atlas = image.load_img(atlas_path)
print(atlas.shape)

plotting.view_img(atlas)   # interactive — drag through it

# brain ATLAS of subject 01

ValueError: File not found: '/content/drive/MyDrive/Neuromatch/Algonauts/algonauts_2025.competitors/fmri/sub-01/atlas/sub-01_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-dseg_parcellation.nii.gz'

In [ ]:
plotting.plot_roi(atlas, title='Schaefer 1000 parcels', cmap='tab20')

⬆ think why Schaefer and not another one

In [39]:
import numpy as np
from nilearn import image

# Correcting the base path to use the previously defined ROOT variable
# The ROOT variable holds the path '/content/drive/MyDrive/Neuromatch/algonauts_2025_challenge_tutorial_data/algonauts_2025.competitors'
base = f'{ROOT}/fmri'
subs = ['sub-01', 'sub-02', 'sub-03', 'sub-05']

atlases = {}
for s in subs:
    p = f'{base}/{s}/atlas/{s}_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-dseg_parcellation.nii.gz'
    atlases[s] = image.load_img(p).get_fdata()
    print(s, atlases[s].shape, int(atlases[s].max()))

# are they the same?
print('01 == 02:', np.array_equal(atlases['sub-01'], atlases['sub-02']))
print('01 == 03:', np.array_equal(atlases['sub-01'], atlases['sub-03']))
print('01 == 05:', np.array_equal(atlases['sub-01'], atlases['sub-05']))

# atlases of all participants are identical - not need to check for each of them

sub-01 (97, 115, 97) 1000
sub-02 (97, 115, 97) 1000
sub-03 (97, 115, 97) 1000
sub-05 (97, 115, 97) 1000
01 == 02: True
01 == 03: True
01 == 05: True


In [41]:
import os
base = f'{ROOT}/fmri'
print(os.listdir(f'{base}/sub-01/atlas/'))

['sub-01_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-dseg_parcellation.nii.gz']


In [42]:
from nilearn import datasets

schaefer = datasets.fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=7, resolution_mm=2)

labels = list(schaefer.labels)
print(len(labels))
print(labels[:5])
print(labels[500:505])

[fetch_atlas_schaefer_2018] Added README.md to /root/nilearn_data

[fetch_atlas_schaefer_2018] Dataset created in /root/nilearn_data/schaefer_2018

[fetch_atlas_schaefer_2018] Downloading data from 
https://raw.githubusercontent.com/ThomasYeoLab/CBIG/v0.14.3-Update_Yeo2011_Schaefer2018_labelname/stable_projects/b
rain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/Schaefer2018_1000Parcels_7Networks_order.txt ...

[fetch_atlas_schaefer_2018]  ...done. (0 seconds, 0 min)

[fetch_atlas_schaefer_2018] Downloading data from 
https://raw.githubusercontent.com/ThomasYeoLab/CBIG/v0.14.3-Update_Yeo2011_Schaefer2018_labelname/stable_projects/b
rain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_2mm
.nii.gz ...

[fetch_atlas_schaefer_2018]  ...done. (0 seconds, 0 min)

1001
['Background', '7Networks_LH_Vis_1', '7Networks_LH_Vis_2', '7Networks_LH_Vis_3', '7Networks_LH_Vis_4']
['7Networks_LH_Default_PHC_3', '7Networks_RH_Vis_1', '7Networks_RH_Vis_2', '7Networks_RH_Vis_3', '7Networks_RH_Vis_4']


meaning of the output ⬆

```[fetch_atlas_schaefer_2018] Dataset directory found: /root/nilearn_data/schaefer_2018

1001
['Background', '7Networks_LH_Vis_1', '7Networks_LH_Vis_2', '7Networks_LH_Vis_3', '7Networks_LH_Vis_4']
['7Networks_LH_Default_PHC_3', '7Networks_RH_Vis_1', '7Networks_RH_Vis_2', '7Networks_RH_Vis_3', '7Networks_RH_Vis_4']
```

`funct` data has 1000 columbs of numbers, column 417 is a brain region but we don't know which one. So we need labels/names of those areas to know which are language areas as it's of key importance for the study.

So why 1001 entries? It's off by 1. So 1000 columns do not line up with 1001 entries. 'Background' isn't a brain region - it's the 0's in the brain atlas volume (skull etc).

If we missed it then the whole parcel analysis would have been shifted by one. BAD.


In [43]:
labels = list(schaefer.labels)[1:] # dropping the background
print(len(labels)) # must be 1000 and IT IS 1000

1000


In [44]:
labels = list(schaefer.labels)[1:]

# what network exist, and who big is each?
from collections import Counter
nets = [l.split('_')[2] for l in labels]
print(Counter(nets))

# sample across the whole range
for i in range(0,1000,40):
  print(i,labels[i])

Counter({'Default': 212, 'SomMot': 194, 'Vis': 162, 'Cont': 129, 'DorsAttn': 122, 'SalVentAttn': 121, 'Limbic': 60})
0 7Networks_LH_Vis_1
40 7Networks_LH_Vis_41
80 7Networks_LH_Vis_81
120 7Networks_LH_SomMot_40
160 7Networks_LH_SomMot_80
200 7Networks_LH_DorsAttn_Post_29
240 7Networks_LH_SalVentAttn_ParOper_8
280 7Networks_LH_SalVentAttn_Med_12
320 7Networks_LH_Cont_Par_4
360 7Networks_LH_Cont_pCun_1
400 7Networks_LH_Default_Temp_8
440 7Networks_LH_Default_PFC_26
480 7Networks_LH_Default_pCunPCC_16
520 7Networks_RH_Vis_21
560 7Networks_RH_Vis_61
600 7Networks_RH_SomMot_20
640 7Networks_RH_SomMot_60
680 7Networks_RH_SomMot_100
720 7Networks_RH_DorsAttn_Post_37
760 7Networks_RH_SalVentAttn_TempOccPar_16
800 7Networks_RH_SalVentAttn_Med_10
840 7Networks_RH_Limbic_TempPole_16
880 7Networks_RH_Cont_PFCl_16
920 7Networks_RH_Default_Par_9
960 7Networks_RH_Default_PFCdPFCm_5


In [45]:
subtags = sorted({'_'.join(l.split('_')[2:-1]) for l in labels})
for s in subtags:
    print(s)

Cont_Cing
Cont_OFC
Cont_PFCd
Cont_PFCl
Cont_PFCmp
Cont_PFCv
Cont_Par
Cont_Temp
Cont_pCun
Default_PFC
Default_PFCdPFCm
Default_PFCv
Default_PHC
Default_Par
Default_Temp
Default_pCunPCC
DorsAttn_FEF
DorsAttn_Post
DorsAttn_PrCv
Limbic_OFC
Limbic_TempPole
SalVentAttn_FrOperIns
SalVentAttn_Med
SalVentAttn_PFCl
SalVentAttn_ParOper
SalVentAttn_PrC
SalVentAttn_TempOcc
SalVentAttn_TempOccPar
SomMot
Vis
